In [1]:
import os
import numpy as np
import pandas as pd
import cv2
import os
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelBinarizer

import warnings
warnings.filterwarnings("ignore")

In [2]:
# Define the base directory where your folders are located
base_dir = r"C:\Users\Sai charan\Desktop\balanced_augmentation"

# List of folder names (these will be the target labels)
folders = ['MildDemented', 'ModerateDemented', 'NonDemented', 'VeryMildDemented']

# Prepare lists to hold data
image_paths = []
image_arrays = []
targets = []
image_size = (128, 128)
# Loop through each folder
for folder in folders:
    folder_path = os.path.join(base_dir, folder)
    
    # Loop through each image in the folder
    for img_name in os.listdir(folder_path):
        # Get the full image path
        img_path = os.path.join(folder_path, img_name)
        
        # Read the image using OpenCV
        img = cv2.imread(img_path)
        if img is None:
            continue
        
        # Convert the image from BGR (OpenCV default) to RGB (optional, depending on your use case)
        img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        img_resized = cv2.resize(img_rgb, image_size)
        # Append the path, image array, and target label (folder name)
        image_paths.append(img_path)
        image_arrays.append(img_resized)
        targets.append(folder)

# Create a DataFrame
df = pd.DataFrame({
    'image_path': image_paths,
    'image_array': image_arrays,
    'target': targets
})

# Display the DataFrame
df.head()


,image_path,image_array,target
0,C:\Users\Sai charan\Desktop\balanced_augmentat...,"[[[0, 0, 0], [0, 0, 0], [0, 0, 0], [0, 0, 0], ...",MildDemented
1,C:\Users\Sai charan\Desktop\balanced_augmentat...,"[[[0, 0, 0], [0, 0, 0], [0, 0, 0], [0, 0, 0], ...",MildDemented
2,C:\Users\Sai charan\Desktop\balanced_augmentat...,"[[[0, 0, 0], [0, 0, 0], [0, 0, 0], [0, 0, 0], ...",MildDemented
3,C:\Users\Sai charan\Desktop\balanced_augmentat...,"[[[0, 0, 0], [0, 0, 0], [0, 0, 0], [0, 0, 0], ...",MildDemented
4,C:\Users\Sai charan\Desktop\balanced_augmentat...,"[[[0, 0, 0], [0, 0, 0], [0, 0, 0], [0, 0, 0], ...",MildDemented


In [3]:
len(image_paths)

12800

In [4]:
dict_c = {'MildDemented':0, 'ModerateDemented':1, 'NonDemented':2, 'VeryMildDemented':3}
df['target'] = df['target'].map(dict_c)
df.head()

,image_path,image_array,target
0,C:\Users\Sai charan\Desktop\balanced_augmentat...,"[[[0, 0, 0], [0, 0, 0], [0, 0, 0], [0, 0, 0], ...",0
1,C:\Users\Sai charan\Desktop\balanced_augmentat...,"[[[0, 0, 0], [0, 0, 0], [0, 0, 0], [0, 0, 0], ...",0
2,C:\Users\Sai charan\Desktop\balanced_augmentat...,"[[[0, 0, 0], [0, 0, 0], [0, 0, 0], [0, 0, 0], ...",0
3,C:\Users\Sai charan\Desktop\balanced_augmentat...,"[[[0, 0, 0], [0, 0, 0], [0, 0, 0], [0, 0, 0], ...",0
4,C:\Users\Sai charan\Desktop\balanced_augmentat...,"[[[0, 0, 0], [0, 0, 0], [0, 0, 0], [0, 0, 0], ...",0


In [5]:
from sklearn.model_selection import train_test_split
import numpy as np

# Assuming df has already been updated with numeric targets
# Flatten the image arrays and convert them into a numpy array
X = np.array([img.flatten() for img in df['image_array']])  # Flatten each image
y = df['target'].values  # Target labels

# Split the data into training and testing sets (80% training, 20% testing)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Display the shapes of the training and testing data
print(f"X_train shape: {X_train.shape}")
print(f"X_test shape: {X_test.shape}")
print(f"y_train shape: {y_train.shape}")
print(f"y_test shape: {y_test.shape}")


X_train shape: (10240, 49152)
X_test shape: (2560, 49152)
y_train shape: (10240,)
y_test shape: (2560,)


In [6]:
## KNN
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score,classification_report, confusion_matrix

# Initialize the KNN model
knn = KNeighborsClassifier(n_neighbors=3)  # You can adjust the number of neighbors

# Fit the model to the training data
knn.fit(X_train, y_train)

# Make predictions on the test data
y_pred = knn.predict(X_test)

accuracy = accuracy_score(y_test, y_pred)

# Evaluate the model
print(f"Accucary of model:{accuracy}")
print("Confusion Matrix:")
print(confusion_matrix(y_test, y_pred))
print("\nClassification Report:")
print(classification_report(y_test, y_pred))


Accucary of model:0.755859375
Confusion Matrix:
[[427 186   9  43]
 [178 409   4  34]
 [  0   0 635   0]
 [ 86  81   4 464]]

Classification Report:
              precision    recall  f1-score   support

           0       0.62      0.64      0.63       665
           1       0.61      0.65      0.63       625
           2       0.97      1.00      0.99       635
           3       0.86      0.73      0.79       635

    accuracy                           0.76      2560
   macro avg       0.76      0.76      0.76      2560
weighted avg       0.76      0.76      0.76      2560



In [7]:
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

# Initialize the Decision Tree model
decision_tree = DecisionTreeClassifier()

# Fit the model to the training data
decision_tree.fit(X_train, y_train)

# Make predictions
y_pred = decision_tree.predict(X_test)

# Evaluate the model
accuracy = accuracy_score(y_test, y_pred)
print(f"Accuracy of Decision Tree model: {accuracy}")
print("Confusion Matrix:")
print(confusion_matrix(y_test, y_pred))
print("\nClassification Report:")
print(classification_report(y_test, y_pred))


Accuracy of Decision Tree model: 0.58359375
Confusion Matrix:
[[308 176  40 141]
 [177 341   5 102]
 [ 35   7 494  99]
 [ 97  95  92 351]]

Classification Report:
              precision    recall  f1-score   support

           0       0.50      0.46      0.48       665
           1       0.55      0.55      0.55       625
           2       0.78      0.78      0.78       635
           3       0.51      0.55      0.53       635

    accuracy                           0.58      2560
   macro avg       0.58      0.58      0.58      2560
weighted avg       0.58      0.58      0.58      2560



In [6]:
from sklearn.svm import LinearSVC
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

# Normalize the data to improve convergence
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Initialize the LinearSVC model
svm = LinearSVC(max_iter=1000, C=1.0, random_state=42, verbose=0)  # Adjust C for faster execution

# Fit the model to the training data
svm.fit(X_train_scaled, y_train)

# Make predictions
y_pred = svm.predict(X_test_scaled)

# Evaluate the model
accuracy = accuracy_score(y_test, y_pred)
print(f"Accuracy of LinearSVC model: {accuracy}")
print("Confusion Matrix:")
print(confusion_matrix(y_test, y_pred))
print("\nClassification Report:")
print(classification_report(y_test, y_pred))


Accuracy of LinearSVC model: 0.68203125
Confusion Matrix:
[[355 200  26  84]
 [201 308  24  92]
 [  2   1 622  10]
 [ 60  84  30 461]]

Classification Report:
              precision    recall  f1-score   support

           0       0.57      0.53      0.55       665
           1       0.52      0.49      0.51       625
           2       0.89      0.98      0.93       635
           3       0.71      0.73      0.72       635

    accuracy                           0.68      2560
   macro avg       0.67      0.68      0.68      2560
weighted avg       0.67      0.68      0.68      2560



In [9]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

# Initialize the Logistic Regression model
log_reg = LogisticRegression(max_iter=1000)  # Increase max_iter if convergence warning appears

# Fit the model to the training data
log_reg.fit(X_train, y_train)

# Make predictions
y_pred = log_reg.predict(X_test)

# Evaluate the model
accuracy = accuracy_score(y_test, y_pred)
print(f"Accuracy of Logistic Regression model: {accuracy}")
print("Confusion Matrix:")
print(confusion_matrix(y_test, y_pred))
print("\nClassification Report:")
print(classification_report(y_test, y_pred))


Accuracy of Logistic Regression model: 0.678515625
Confusion Matrix:
[[353 221  14  77]
 [218 324   1  82]
 [  7   1 606  21]
 [ 81  87  13 454]]

Classification Report:
              precision    recall  f1-score   support

           0       0.54      0.53      0.53       665
           1       0.51      0.52      0.52       625
           2       0.96      0.95      0.96       635
           3       0.72      0.71      0.72       635

    accuracy                           0.68      2560
   macro avg       0.68      0.68      0.68      2560
weighted avg       0.68      0.68      0.68      2560



In [6]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report


# Initialize the Random Forest modelha
rf_model = RandomForestClassifier(max_depth= 20, min_samples_leaf= 1, min_samples_split= 5, n_estimators= 50,random_state = 32)  # You can adjust parameters as needed

# Fit the model to the training data
rf_model.fit(X_train, y_train)

# Make predictions on the test data
y_pred_rf = rf_model.predict(X_test)

# Calculate accuracy
accuracy_rf = accuracy_score(y_test, y_pred_rf)

print(f"Accuracy: {accuracy_rf:.2f}")


Accuracy: 0.75


In [8]:
from sklearn.naive_bayes import GaussianNB
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

# Initialize the Naive Bayes model
nb = GaussianNB()

# Fit the model to the training data
nb.fit(X_train, y_train)

# Make predictions
y_pred = nb.predict(X_test)

# Evaluate the model
accuracy = accuracy_score(y_test, y_pred)
print(f"Accuracy of Naive Bayes model: {accuracy}")
print("Confusion Matrix:")
print(confusion_matrix(y_test, y_pred))
print("\nClassification Report:")
print(classification_report(y_test, y_pred))


Accuracy of Naive Bayes model: 0.39375
Confusion Matrix:
[[175  69 195 226]
 [231 119   9 266]
 [  0   0 635   0]
 [ 66  42 448  79]]

Classification Report:
              precision    recall  f1-score   support

           0       0.37      0.26      0.31       665
           1       0.52      0.19      0.28       625
           2       0.49      1.00      0.66       635
           3       0.14      0.12      0.13       635

    accuracy                           0.39      2560
   macro avg       0.38      0.39      0.34      2560
weighted avg       0.38      0.39      0.34      2560



In [7]:
from sklearn.ensemble import AdaBoostClassifier
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

# Initialize the AdaBoost model
adaboost = AdaBoostClassifier(n_estimators=50, random_state=42)  # You can adjust n_estimators as needed

# Fit the model to the training data
adaboost.fit(X_train, y_train)

# Make predictions
y_pred = adaboost.predict(X_test)

# Evaluate the model
accuracy = accuracy_score(y_test, y_pred)
print(f"Accuracy of AdaBoost model: {accuracy}")
print("Confusion Matrix:")
print(confusion_matrix(y_test, y_pred))
print("\nClassification Report:")
print(classification_report(y_test, y_pred))


Accuracy of AdaBoost model: 0.48984375
Confusion Matrix:
[[151 332 127  55]
 [133 456   1  35]
 [ 15   0 549  71]
 [ 72 127 338  98]]

Classification Report:
              precision    recall  f1-score   support

           0       0.41      0.23      0.29       665
           1       0.50      0.73      0.59       625
           2       0.54      0.86      0.67       635
           3       0.38      0.15      0.22       635

    accuracy                           0.49      2560
   macro avg       0.46      0.49      0.44      2560
weighted avg       0.46      0.49      0.44      2560

